# 3.4 Drift

**What this notebook is for.** Answering "has the world moved since we fitted
this?" for a model that is already in production. Run it on a schedule, not once.

**Why an unsupervised detector needs this more than a classifier does.** A
supervised model's decay shows up in its metrics, because labels keep arriving.
An anomaly detector has no labels in production. Its scores can drift into
uselessness — everything anomalous, or nothing — and every dashboard will look
normal, because the dashboard is built from the scores that drifted.

**The three measurements.**

- `population_stability_index` — the standard PSI over binned distributions. It
  needs the *training* bin edges, not edges recomputed on the current data;
  recomputing the edges is how you get a PSI of zero for a distribution that
  moved entirely, because the bins moved with it. `score_drift` handles that for
  you.
- `total_variation_distance` — a bounded companion to PSI. PSI is unbounded and
  numerically unstable when a bin empties; TVD is neither, so disagreement
  between the two is usually a near-empty bin and worth looking at directly.
- `cluster_drift` — for the clustering models only. Compares cluster occupancy
  and distance-to-centroid between fit time and now. A cluster that has emptied
  or a mean distance that has doubled means the geometry the scores were defined
  in has changed, which is more serious than the score distribution shifting.

**Thresholds.** `DriftThresholds` holds the cut points and `DRIFT_STATUSES` the
resulting labels. They are conventional, not derived — treat a first `warn` as a
prompt to look, not as an alarm, and calibrate them against a few months of your
own history before wiring them to a pager.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

In [ ]:
from trust_score_05.ml.config import load_ml_config
from trust_score_05.ml.drift import DRIFT_STATUSES, DriftThresholds

ENV = "dev"
cfg = load_ml_config(env=ENV)

thresholds = DriftThresholds()
print("thresholds:", thresholds)
print("statuses  :", DRIFT_STATUSES)

In [ ]:
from trust_score_05.ml.jobs.base import build_spark_session

spark = build_spark_session("notebook")
spark.sparkContext.setLogLevel("WARN")
spark

## Load the champion

The bundle carries everything: features, preprocessing id, impute rules, fitted
preprocessor, fitted model. That is what makes drift checkable at all — without
the fitted preprocessor you cannot put current data into the same space the
training scores were computed in, and comparing scores across two different
spaces is meaningless.

In [ ]:
from trust_score_05.common.io.serialization import load_bundle

BUNDLE_URI = None  # e.g. f"{cfg.base_path}/runs/.../model_bundle.joblib"

if BUNDLE_URI is None:
    raise SystemExit(
        "Set BUNDLE_URI to the champion bundle from 3.3-evaluate-and-compare.ipynb. "
        "config_prefix(run_prefix(...), hyperparam_id(...)) builds the path."
    )

bundle = load_bundle(BUNDLE_URI)
features = bundle.features
print("model            :", bundle.model_name)
print("features         :", len(features))
print("preprocessing id :", bundle.preprocessing_id)
print("format version   :", bundle.format_version)

## Reconstruct the training reference

In [ ]:
from trust_score_05.common.splitting import remove_fraud_customers
from trust_score_05.ml.datasets import bounded_to_pandas, load_fraud_customer_ids, load_training_matrix

training = load_training_matrix(spark, cfg, features=features)
training = remove_fraud_customers(training, load_fraud_customer_ids(spark, cfg))
train_pdf = bounded_to_pandas(training, max_rows=cfg.max_training_rows)

train_matrix = bundle.preprocessor.transform(train_pdf[features])
train_scores = bundle.model.score(train_matrix)

print("training rows :", len(train_pdf))
print("score range   :", float(train_scores.min()), "->", float(train_scores.max()))

## The current population

Point this at a recent snapshot. It must be scored through the *bundle's*
preprocessor, not a freshly fitted one — a refitted scaler would absorb the very
shift you are trying to measure and report no drift at all.

In [ ]:
EVAL_PATH = None  # a recent feature-matrix snapshot

if EVAL_PATH is None:
    print("EVAL_PATH is None -- reusing the training frame as a no-drift control.")
    eval_pdf = train_pdf
else:
    from trust_score_05.ml.datasets import read_parquet_path

    eval_pdf = bounded_to_pandas(
        read_parquet_path(spark, EVAL_PATH, features=features),
        max_rows=cfg.max_scoring_rows,
    )

eval_matrix = bundle.preprocessor.transform(eval_pdf[features])
eval_scores = bundle.model.score(eval_matrix)
print("evaluation rows:", len(eval_pdf))

## Score drift

In [ ]:
from trust_score_05.ml.drift import score_drift

result = score_drift(train_scores, eval_scores, thresholds=thresholds)
for key, value in sorted(vars(result).items() if hasattr(result, "__dict__") else result.items()):
    print(f"{key:28s} {value}")

In [ ]:
import matplotlib.pyplot as plt

from trust_score_05.common.viz.plots import configure_matplotlib

configure_matplotlib()

figure, axis = plt.subplots(figsize=(9, 4))
axis.hist(train_scores, bins=60, alpha=0.55, density=True, label="training (reference)")
axis.hist(eval_scores, bins=60, alpha=0.55, density=True, label="current")
axis.set_title(f"{bundle.model_name} -- score distribution, then and now")
axis.set_xlabel("anomaly score")
axis.set_ylabel("density")
axis.legend()
plt.tight_layout()
plt.show()

## Per-feature drift

A shifted score distribution is a symptom; the per-feature PSI tells you which
input moved. Sort descending and look at the top few — usually one or two
features carry almost all of it, and usually the reason is upstream rather than
in the model.

In [ ]:
import numpy as np
import pandas as pd

from trust_score_05.ml.drift import population_stability_index, total_variation_distance

rows = []
for name in features:
    reference = train_pdf[name].dropna().to_numpy(dtype=float)
    current = eval_pdf[name].dropna().to_numpy(dtype=float)
    if len(reference) < 100 or len(current) < 100:
        continue
    edges = np.quantile(reference, np.linspace(0, 1, 11))
    edges = np.unique(edges)
    if len(edges) < 3:
        continue
    rows.append(
        {
            "feature": name,
            "psi": population_stability_index(reference, current, edges=edges),
            "tvd": total_variation_distance(reference, current),
            "train_null_rate": float(train_pdf[name].isna().mean()),
            "eval_null_rate": float(eval_pdf[name].isna().mean()),
        }
    )

per_feature = pd.DataFrame(rows).sort_values("psi", ascending=False)
print(f"{len(per_feature)} feature(s) with enough data to compare")
per_feature.head(25)

## Null-rate drift, which PSI will not catch

PSI is computed on the non-null values. A feature that went from 2% null to 60%
null can have a perfectly stable PSI while having become useless, because the
imputer is now supplying most of its values. Check this column explicitly.

In [ ]:
per_feature["null_rate_delta"] = (
    per_feature["eval_null_rate"] - per_feature["train_null_rate"]
)
per_feature.reindex(
    per_feature["null_rate_delta"].abs().sort_values(ascending=False).index
).head(20)[["feature", "train_null_rate", "eval_null_rate", "null_rate_delta", "psi"]]

## Cluster drift, for the clustering models

Only meaningful for `kmeans`, `weighted_kmeans` and `hdbscan`. For the others the
concept does not apply and the call will tell you so.

In [ ]:
from trust_score_05.ml.drift import CLUSTER_DRIFT_COLUMNS, cluster_drift

print("cluster drift columns:", CLUSTER_DRIFT_COLUMNS)

if bundle.model_name in {"kmeans", "weighted_kmeans", "hdbscan"}:
    train_labels = bundle.model.labels(train_matrix)
    train_distances = bundle.model.distances(train_matrix)
    eval_labels = bundle.model.labels(eval_matrix)
    eval_distances = bundle.model.distances(eval_matrix)

    clusters = cluster_drift(
        train_labels, train_distances, eval_labels, eval_distances, thresholds=thresholds
    )
    display(clusters)
else:
    print(f"{bundle.model_name} is not a clustering model -- skipping.")

## The consolidated report

In [ ]:
from trust_score_05.ml.drift import drift_report
from trust_score_05.ml.evaluation import score_population

scored = score_population(eval_pdf, bundle.preprocessor, bundle.model)

report = drift_report(
    adapter=bundle.model,
    train_matrix=train_matrix,
    scored=scored,
    eval_matrix=eval_matrix,
    train_scores=train_scores,
    thresholds=thresholds,
)
report

## Reading the result

- **ok** — nothing to do. Record the numbers so the trend exists.
- **warn** — look at the top per-feature PSI and the null-rate deltas. Usually one
  upstream feed changed. Fix it there.
- **alert** — do not silently keep scoring. Either the population genuinely moved,
  in which case refit, or something upstream broke, in which case refitting bakes
  the breakage in. Find out which before doing either.

**Schedule this.** Drift found six months late is drift that has been affecting
decisions for six months.